In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"

In [0]:
cast_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/netflix_cast")

In [0]:
cast_df.display()

In [0]:
silver_cast = (
    cast_df
    .withColumn("show_id", trim(col("show_id")))
    .withColumn("cast", trim(col("cast")))
    .withColumn(
        "cast",
        when(col("cast") == "", None)
         .otherwise(col("cast"))
    )
    .filter(col("show_id").isNotNull())
    .filter(col("cast").isNotNull())
    .dropDuplicates(["show_id", "cast"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)



In [0]:
silver_cast.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/netflix_cast")